# Fire analysis


In [1]:
DB_FILE = "fire.duckdb"
STATE = "OR"

## DuckDB / SQL Magic setup

Using [JupySQL](https://jupysql.readthedocs.io/). [Docs from DuckDB.](https://duckdb.org/docs/current/guides/python/jupyter)


In [2]:
%config SqlMagic.autopandas = True
%config SqlMagic.displaycon = False

In [3]:
import duckdb

%load_ext sql
conn = duckdb.connect(DB_FILE)
%sql conn --alias duckdb

Tip: You may define configurations in /Users/afeld/dev/ptps-wildfire-demo/pyproject.toml or /Users/afeld/.jupysql/config.

Did not find user configurations in /Users/afeld/dev/ptps-wildfire-demo/pyproject.toml.

In [4]:
from helpers import run_script_in_db


run_script_in_db(conn, "setup.sql")

## State-level


In [5]:
%%sql
SELECT geom
FROM state_boundaries
WHERE STUSPS = '{{STATE}}';

,geom
0,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 254, 0, 0, 0, 13, ..."


In [6]:
%%sql
SELECT DISTINCT state.stusps
FROM state_boundaries AS state
INNER JOIN burn_prob_1km AS bp
    ON ST_Contains(state.geom, bp.point)
ORDER BY state.stusps;

,STUSPS
0,CA
1,OR
2,WA


In [7]:
%%sql
SELECT COUNT(*)
FROM burn_prob_1km AS bp
INNER JOIN state_boundaries AS state
    ON ST_Contains(state.geom, bp.point)
WHERE state.stusps = '{{STATE}}';

,count_star()
0,2599


In [8]:
%%sql
SELECT
    min(bp_2011),
    max(bp_2011)
FROM burn_prob_1km;

,min(bp_2011),max(bp_2011)
0,0.0,0.017974


In [9]:
import matplotlib
from lonboard import Map, ScatterplotLayer
from lonboard.colormap import apply_continuous_cmap

query = """
SELECT
    point,
    bp_2011
FROM burn_prob_1km
WHERE bp_2011 > 0.0;
"""

values = conn.execute(query).df()["bp_2011"]

vmin = values.min()
vmax = values.max()
scaled = (values - vmin) / (vmax - vmin)

# https://matplotlib.org/stable/gallery/color/colormap_reference.html
cmap = matplotlib.colormaps["OrRd"]
colors = apply_continuous_cmap(scaled, cmap)

layer = ScatterplotLayer.from_duckdb(
    query,
    conn,
    crs="EPSG:4326",
    get_fill_color=colors,
    radius_min_pixels=2,
)
m = Map(layer)
m

## Active fires


### Query data

> Each MODIS active fire/thermal hotspot location represents the center of a 1km pixel that is flagged by the algorithm as containing one or more fires within the pixel.

https://www.earthdata.nasa.gov/data/tools/firms

https://firms.modaps.eosdis.nasa.gov/active_fire/#firms-txt


In [10]:
%%sql
SELECT *
FROM active_fires
LIMIT 10;


,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,confidence,version,bright_t31,frp,daynight
0,19.63446,-71.10041,307.92,1.02,1.01,2026-08-08,0120,T,54,6.1NRT,283.25,6.52,N
1,20.68952,-76.79361,305.99,1.57,1.24,2026-08-08,0120,T,61,6.1NRT,291.72,11.11,N
2,20.68712,-76.80814,302.85,1.58,1.24,2026-08-08,0120,T,39,6.1NRT,291.28,7.13,N
3,36.31087,-78.26340,309.82,1.09,1.04,2026-08-08,0124,T,77,6.1NRT,293.68,8.00,N
4,18.06466,-66.39038,311.41,1.73,1.29,2026-08-08,0120,T,75,6.1NRT,293.85,19.73,N
5,18.06106,-71.67026,307.81,1.00,1.00,2026-08-08,0120,T,68,6.1NRT,295.54,5.26,N
6,18.24512,-70.45757,304.89,1.03,1.02,2026-08-08,0120,T,57,6.1NRT,294.27,3.45,N
7,40.89050,-77.71103,302.02,1.00,1.00,2026-08-08,0124,T,17,6.1NRT,288.12,5.76,N
8,41.45594,-81.67901,308.53,1.27,1.12,2026-08-08,0126,T,75,6.1NRT,289.14,9.68,N
9,41.45945,-81.67040,302.45,1.27,1.12,2026-08-08,0126,T,11,6.1NRT,288.69,4.30,N
